##### ARTI 560 - Computer Vision

## Visual Representations with DINOv2 - Exercise

### Exercise 1: Unsupervised Clustering

In this exercise, you will use the `KMeans` algorithm from sklearn to group 20 images from the Oxford Pet dataset into 2 clusters (Cats vs. Dogs) based purely on their CLS tokens.

Instructions:

1.  Extract the 384-dimensional [CLS] tokens from 20 images of the Oxford-IIIT Pet dataset. Ensure your selection includes a mix of both cats and dogs.

2. Apply K-Means Clustering ($n=2$) to group the vectors based on mathematical similarity rather than provided labels.

3. Compare the predicted clusters against ground-truth labels.

In [9]:
import glob
import torch
import numpy as np
from PIL import Image
from transformers import AutoImageProcessor, AutoModel
from sklearn.cluster import KMeans
from sklearn.metrics import classification_report, accuracy_score

MODEL_ID = "facebook/dinov2-small"
device = "cuda" if torch.cuda.is_available() else "cpu"

processor = AutoImageProcessor.from_pretrained(MODEL_ID)
model = AutoModel.from_pretrained(MODEL_ID).to(device)
model.eval()

cat_paths = glob.glob("test_images/cat/*.jpg")
dog_paths = glob.glob("test_images/dog/*.jpg")
all_paths = cat_paths + dog_paths

y_true = [0] * len(cat_paths) + [1] * len(dog_paths)

embeddings = []
for path in all_paths:
    img = Image.open(path).convert("RGB")
    inputs = processor(images=img, return_tensors="pt").to(device)
    
    with torch.no_grad():
        outputs = model(**inputs)
        cls_token = outputs.last_hidden_state[:, 0].cpu().numpy()
        embeddings.append(cls_token.squeeze(0))  # FIX: squeeze to (384,)

embeddings = np.vstack(embeddings)  # FIX: stack into (20, 384) array

kmeans = KMeans(n_clusters=2, random_state=42)
y_pred = kmeans.fit_predict(embeddings)

acc = max(accuracy_score(y_true, y_pred), accuracy_score(y_true, 1 - y_pred))
print("Clustering Accuracy:", acc)
print("\nClassification Report:\n", classification_report(y_true, y_pred))

Clustering Accuracy: 0.95

Classification Report:
               precision    recall  f1-score   support

           0       1.00      0.90      0.95        10
           1       0.91      1.00      0.95        10

    accuracy                           0.95        20
   macro avg       0.95      0.95      0.95        20
weighted avg       0.95      0.95      0.95        20



c:\Users\leena\anaconda3\envs\ImageProcessinng\lib\site-packages\sklearn\cluster\_kmeans.py:1416: FutureWarning: The default value of `n_init` will change from 10 to 'auto' in 1.4. Set the value of `n_init` explicitly to suppress the warning
  super()._check_params_vs_input(X, default_n_init=10)
c:\Users\leena\anaconda3\envs\ImageProcessinng\lib\site-packages\sklearn\cluster\_kmeans.py:1440: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=1.
  warnings.warn(


### Exercise 2: Image Classification with DINOv2

In this exercise you'll use a DINOv2 model with a pre-trained linear head to classify an image. You will observe how the model maps visual features to specific ImageNet-1k categories.

Instructions:
1. For this exercise, you must use the following Model ID. This specific checkpoint includes the necessary classification head trained on ImageNet-1k:

    Model ID: `facebook/dinov2-small-imagenet1k-1-layer`

2. Find an image online to make the inference. To ensure the model has a fair chance of success, the image should belong to one of the ImageNet-1k classes (e.g., a Golden Retriever, a grand piano, a school bus, or a coffee mug).

In [ ]:
from transformers import AutoImageProcessor, AutoModelForImageClassification
import torch
from PIL import Image

MODEL_ID = "facebook/dinov2-small-imagenet1k-1-layer"
device = "cuda" if torch.cuda.is_available() else "cpu"

processor = AutoImageProcessor.from_pretrained(MODEL_ID)
model = AutoModelForImageClassification.from_pretrained(MODEL_ID)

# Load single test image (Golden Retriever)
img = Image.open("test_images/dog/dog3.jpg").convert("RGB")
inputs = processor(images=img, return_tensors="pt")

#run Inference
with torch.no_grad():
    outputs = model(**inputs)
    logits = outputs.logits
    predicted_class_idx = logits.argmax(-1).item()

# Map to human-readable class name
predicted_label = model.config.id2label[predicted_class_idx]
print(f"The model classified this image as: {predicted_label}")

The model classified this image as: golden retriever
